1. Szukanie danych oraz ich przygotowanie:

1.1 Cel i etapy wstępne:
Celem tego etapu pracowni rotacyjnej było zgromadzenie, ujednolicenie oraz wstępna analiza statystyczna danych dotyczących ekotoksyczności substancji zmniejszających palność (ang. flame retardants). Pierwszy etap prac obejmował przegląd literatury naukowej w bazach takich jak PubMed, Web of Science oraz Scopus w celu identyfikacji związków należących do tej grupy oraz zebrania dostępnych informacji o ich strukturze i profilu toksykologicznym.

W wyniku selekcji wyłoniono grupę ponad 60 kluczowych substancji, dla których utworzono bazę danych zawierającą podstawowe identyfikatory chemiczne, takie jak numery CAS oraz notację SMILES (brakujące dane zostały uzupełnione ręcznie).

2. Przetwarzanie danych i standaryzacja:

Zgromadzone dane strukturalne posłużyły do wygenerowania zestawu deskryptorów molekularnych reprezentujących właściwości fizykochemiczne analizowanych związków.

2.1 Następnie przystąpiono do usuwania duplikatów za pomocą wbudowanej funkcji w excelu, co dało nam ostateczny wynik 41 związków. 

2.2 Kolejnym krokiem było obliczenie masy molowej 

2.3 oraz obliczenie brakujących wartości pEC50_standardized wg wzoru (ręcznie w excelu): 
pEC50 = -log_10 * (raw_value_mg_L / 1000 * mol_weight_g_mol)

Plik ze wszystkimi informacjami został zapisany jako deskryptory_z_cas_2.xlsx


Obliczanie masy molowej:

In [4]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

def calculate_missing_mw(row):
    current_mw = pd.to_numeric(row['mol_weight_g_mol'], errors='coerce')

    if pd.notna(current_mw) and current_mw > 0:
        return current_mw
    
    smiles = str(row['smiles']).strip() if pd.notna(row['smiles']) else ""

    if not smiles or smiles.lower() == "nan":
        return None
        
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return Descriptors.MolWt(mol)
        else:
            return None
    except:
        return None

file_path = 'Baza_FR_zajęciawszystkieSMILES.xlsx'

df = pd.read_excel(file_path, sheet_name='Unified_Units_pEC50', skiprows=3)
print("Wczytane kolumny:", df.columns.tolist())
df['mol_weight_g_mol'] = df.apply(calculate_missing_mw, axis=1)

output_name = 'Baza_FR_wynik.xlsx'

df.to_excel(output_name, index=False, sheet_name="Unified_Units_pEC50")

print(f"\nPlik zapisany jako: {output_name}")
print(df[['cas', 'smiles', 'mol_weight_g_mol']].head())

Wczytane kolumny: ['record_type', 'source_group', 'source_detail', 'compound_label', 'cas', 'smiles', 'organism', 'endpoint', 'duration_h', 'raw_value_mg_L', 'raw_value_neglog_mol_L', 'mol_weight_g_mol', 'ec50_mol_L_standardized', 'pEC50_standardized', 'preferred_modeling_unit', 'conversion_basis', 'backcalc_mg_L_from_standardized', 'fr_relevance_flag', 'note', 'Unnamed: 19', 'Converted from existing -log10(mol/L)', 34]

Plik zapisany jako: Baza_FR_wynik.xlsx
        cas                                       smiles  mol_weight_g_mol
0  101-55-3                  BrC1=CC=C(OC=2C=CC=CC2)C=C1           249.107
1  101-84-8                    O(C=1C=CC=CC1)C=2C=CC=CC2           170.211
2  106-41-2                             BrC1=CC=C(O)C=C1           173.009
3  115-86-6  O=P(OC=1C=CC=CC1)(OC=2C=CC=CC2)OC=3C=CC=CC3           326.288
4  115-86-6  O=P(OC=1C=CC=CC1)(OC=2C=CC=CC2)OC=3C=CC=CC3           326.288


3. Analiza eksploracyjna danych:

 Wraz z przygotowaną macierzą z uzupełnionymi danymi przystąpiono do dalszej analizy. Z uwagi na wielowymiarowość deskryptorów, w dalszej analizie posłużono się:

3.1. Hierarchiczną Analizę Skupień (HCA) – w celu zbadania podobieństwa strukturalno-toksykologicznego między związkami FR i przedstawienia ich relacji w postaci dendrogramu (wyniki zostały zapisane w pliku o nazwie HCA.ipynb)

3.2. Analizą Składowych Głównych (PCA) - w celu zredukowania wymiarowości, zredukowania liczby badanych cech bez utraty istotnych informacji fizykochemicznych (wyniki zostały zapisane w pliku o nazwie PCA.ipynb)
